In Lecture 20 of mcs 320, we apply memoization to make recursive functions more efficient.

# 1. Recursive Functions

The Fibonacci numbers are defined as $F(0) = 0$, $F(1) = 1$,
and for $n > 1$: $F(n) = F(n-1) + F(n-2)$.

Below is a basic Python function, that follows the recursive definition.

In [1]:
def F(n):
    """
    Returns the n-th Fibonacci number.
    """
    if n == 0:
        return 0
    elif n == 1:
        return 1
    else:
        return F(n-1) + F(n-2)

Let us look at the first ten Fibonacci numbers.

In [2]:
[F(k) for k in range(10)]

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

The main problem with the definition is that it is too inefficient.

In [3]:
timeit('F(20)')

25 loops, best of 3: 7.92 ms per loop

In [4]:
timeit('F(21)')

25 loops, best of 3: 13 ms per loop

In [5]:
timeit('F(22)')

25 loops, best of 3: 21.6 ms per loop

In [6]:
timeit('F(23)')

25 loops, best of 3: 34.5 ms per loop

Just to compute the next number in the sequence takes significantly more time.

The reason for this high cost is the repeated function calls.
Consider the computation of $F(2) = F(0) + F(1) = 0 + 1$.
As both $F(0)$ and $F(1)$ are base cases in the recursion,
the calls $F(0)$ and $F(1)$ are leaves of the binary tree of calls.

In [7]:
L0 = LabelledBinaryTree([], label='F(0)')
L1 = LabelledBinaryTree([], label='F(1)')
L2 = LabelledBinaryTree([L0, L1], label='F(2)')
ascii_art(L2)

  _F(2)__
 /       \
F(0)      F(1)

Consider $F(3)$, $F(4)$, and $F(5)$.

In [8]:
L3 = LabelledBinaryTree([L1, L2], label='F(3)')
L4 = LabelledBinaryTree([L2, L3], label='F(4)')
L5 = LabelledBinaryTree([L3, L4], label='F(5)')
ascii_art(L5)

       _______________F(5)_______________
      /                                  \
  ___F(3)___                       _______F(4)________
 /          \                     /                   \
F(1)        _F(2)__             _F(2)__             ___F(3)___
           /       \           /       \           /          \
          F(0)      F(1)      F(0)      F(1)      F(1)        _F(2)__
                                                             /       \
                                                            F(0)      F(1)

From the tree of function calls, we see many repeated instances
of $F(0)$ and $F(1)$.  Observe too that $F(3)$ is computed twice, 
and $F(2)$ is computed three times.

# 2. Memoization in Python

Memoization is a technique to speed up programs by storing the results of function calls.

We will memoize the Fibonacci function $F$,
using a dictionary as an extra argument of the function
to store the results of the function calls.

In [9]:
def memoizedF(n, D={}):
    """
    All calls made to memoizedF() are stored in D.
    """
    if n in D:       # dictionary lookup
        return D[n]
    else:
        if n == 0:
            result = 0
        elif n == 1:
            result = 1
        else:
            result = memoizedF(n-1) + memoizedF(n-2)
        D[n] = result # store the result in the dictionary
        return result

Let us check the computation of the first 10 Fibonacci numbers again.

In [10]:
[memoizedF(k) for k in range(10)]

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

In [11]:
timeit('memoizedF(100)')

625 loops, best of 3: 217 ns per loop

# 3. Memoization in SageMath

Consider the Chebyshev polynomials, defined as
$T(0,x)=1$, $T(1,x) = x$ and for degree $n > 1$:
$T(n,x) = 2 x T(n−1,x) − T(n−2,x)$.

The Chebyshev polynomials are already available.

In [13]:
[chebyshev_T(k,x) for k in range(10)]

[1,
 x,
 2*x^2 - 1,
 4*x^3 - 3*x,
 8*x^4 - 8*x^2 + 1,
 16*x^5 - 20*x^3 + 5*x,
 32*x^6 - 48*x^4 + 18*x^2 - 1,
 64*x^7 - 112*x^5 + 56*x^3 - 7*x,
 128*x^8 - 256*x^6 + 160*x^4 - 32*x^2 + 1,
 256*x^9 - 576*x^7 + 432*x^5 - 120*x^3 + 9*x]

Let us define our own, efficient definition of the 
Chebyshev polynomials, with the definition of a function ``T``.

Because Chebyshev polynomials are defined by two parameters:

1. ``n`` is the degree of the polynomial,

2. ``x`` is the symbol of the variable in the polynomial,

the keys of the dictionary are tuples ``(n, x)``.

In [14]:
def T(n, x, D={}):
    """
    Memoized Chebyshev function of degree n in x.
    """
    if (n, x) in D:
        return D[(n, x)]
    else:
        if n == 0:
            result = 1
        elif n == 1:
            result = x
        else:
            result = expand(2*x*T(n-1,x) - T(n-2,x)) # normalize
        D[(n, x)] = result # store the normalized result
        return result

Observe the normalization of the Chebyshev polynomial.

In [15]:
[T(k, x) for k in range(10)]

[1,
 x,
 2*x^2 - 1,
 4*x^3 - 3*x,
 8*x^4 - 8*x^2 + 1,
 16*x^5 - 20*x^3 + 5*x,
 32*x^6 - 48*x^4 + 18*x^2 - 1,
 64*x^7 - 112*x^5 + 56*x^3 - 7*x,
 128*x^8 - 256*x^6 + 160*x^4 - 32*x^2 + 1,
 256*x^9 - 576*x^7 + 432*x^5 - 120*x^3 + 9*x]

In [16]:
timeit('T(100, x)')

625 loops, best of 3: 474 ns per loop

As before, we see that this memoized version is very efficient.